# Ground states as matrix product states

`MPSGSOperator` is a density operator $\rho = |\psi_0\rangle\langle\psi_0|$ whose state is the ground state of a Hamiltonian, stored as a matrix product state (MPS) computed with a DMRG. It gives `expect` and `partial_trace` for systems that are too large for exact diagonalization.

This notebook shows how to use it, how to choose the order of the sites along the chain, how to check the accuracy, and how to compare it with Gibbs states at low temperature. See the user guide (*Ground states as matrix product states*) for the details and limitations.

> The MPS ground-state operator is experimental, and the ground state is assumed to be non-degenerate.

In [ ]:
import time

import numpy as np
from scipy.sparse.linalg import eigsh

from qalma import build_system
from qalma.mps import MPS, MPSGSOperator
from qalma.operators.states import GibbsDensityOperator

## 1. A small chain, checked against exact diagonalization

The ground state of the Heisenberg chain with open boundaries and 12 spins. The argument `permutation` gives the order of the sites along the chain; here we list the names explicitly. The DMRG runs the first time the energy is needed.

In [ ]:
n_sites = 8
system = build_system("open chain lattice", "spin", L=n_sites)
H = system.global_operator("Hamiltonian")
order = [f"1[{i}]" for i in range(n_sites)]

rho = MPSGSOperator(H, permutation=order, chi_max=32)
print("DMRG energy:", rho.energy)
print(rho.dmrg_info["converged"], rho.dmrg_info["sweeps"], rho.dmrg_info["max_truncation"])
print(rho)

In [ ]:
block = tuple(sorted(system.sites))
e_exact = eigsh(H.to_qutip(block).data_as("csr_matrix"), k=1, which="SA")[0][0]
print("exact energy:", e_exact, " difference:", rho.energy - e_exact)

Expectation values of operators of any support are computed from the MPS, with no dense matrix of the full system. The cost of the reduced state of a few sites is exponential in the number of sites kept, but only linear in the distance between them along the chain.

In [ ]:
Sz = lambda i: system.site_operator("Sz", f"1[{i}]")

for r in range(1, 6):
    print(r, np.real(rho.expect(Sz(0) * Sz(r))))

In [ ]:
reduced = rho.partial_trace(frozenset({"1[5]", "1[6]"}))
print(type(reduced).__name__)
print(reduced.to_qutip(("1[5]", "1[6]")))

## 2. Beyond exact diagonalization

An open chain of 40 spins has $2^{40}$ states. The ground state with bond dimension 8 takes a few seconds.

In [ ]:
n_big = 40
big = build_system("open chain lattice", "spin", L=n_big)
H_big = big.global_operator("Hamiltonian")
natural = [f"1[{i}]" for i in range(n_big)]

start = time.time()
rho_big = MPSGSOperator(H_big, permutation=natural, chi_max=8)
print("E0 =", rho_big.energy, " E0/N =", rho_big.energy / n_big)
print("converged:", rho_big.dmrg_info["converged"],
      " sweeps:", rho_big.dmrg_info["sweeps"],
      " truncation:", rho_big.dmrg_info["max_truncation"])
print("time: %.1f s" % (time.time() - start))

In [ ]:
# Spin-spin correlations from the middle of the chain
center = n_big // 2
for r in (1, 2, 3, 4, 8):
    value = rho_big.expect(big.site_operator("Sz", f"1[{center}]")
                           * big.site_operator("Sz", f"1[{center + r}]"))
    print(r, np.real(value))

## 3. The order of the sites matters

By default the chain follows the lexicographic order of the site names, which for a chain with more than ten sites is `1[0], 1[10], 1[11], ..., 1[1], 1[2], ...`. The state is the same if the bond dimension is large enough, but the DMRG needs more sweeps, a larger truncation and more time. Here, with 20 spins and the same bond dimension:

In [ ]:
n_ord = 20
sys_ord = build_system("open chain lattice", "spin", L=n_ord)
H_ord = sys_ord.global_operator("Hamiltonian")

for label, perm in (("natural", [f"1[{i}]" for i in range(n_ord)]),
                    ("lexicographic", None)):
    start = time.time()
    state = MPSGSOperator(H_ord, permutation=perm, chi_max=8,
                          dmrg_options={"max_sweeps": 10})
    energy = state.energy
    info = state.dmrg_info
    print(f"{label:14s} E0 = {energy:.8f}  sweeps = {info['sweeps']:2d}  "
          f"converged = {info['converged']}  truncation = {info['max_truncation']:.1e}  "
          f"({time.time() - start:.1f} s)")

## 4. Gibbs states at low temperature

The reduced state of a Gibbs state tends to the one of the ground state when $\beta\to\infty$. For the bond term $S^z_3 S^z_4$ of an open chain with 8 spins, the exact Gibbs states (dense) and the MPS give:

In [ ]:
n_g = 8
sys_g = build_system("open chain lattice", "spin", L=n_g)
H_g = sys_g.global_operator("Hamiltonian")
obs = sys_g.site_operator("Sz", "1[3]") * sys_g.site_operator("Sz", "1[4]")
sites = frozenset({"1[3]", "1[4]"})

gs = MPSGSOperator(H_g, permutation=[f"1[{i}]" for i in range(n_g)])
print("ground state:", np.real(gs.partial_trace(sites).expect(obs)))
for beta in (1, 2, 4, 8, 16, 32):
    gibbs = GibbsDensityOperator((beta * H_g).simplify())
    print(f"beta = {beta:2d}:", np.real(gibbs.partial_trace(sites).expect(obs)))

## 5. Using an MPS computed elsewhere

The DMRG of QALMA is plain. A state computed with another code can be wrapped with `from_mps`, as long as its positions follow the same `permutation`. As an illustration, we build the MPS of the exact ground state of the 8-site chain from its state vector.

In [ ]:
block = tuple(sorted(sys_g.sites))          # '1[0]', ..., '1[7]': the natural order
vals, vecs = eigsh(H_g.to_qutip(block).data_as("csr_matrix"), k=1, which="SA")
mps = MPS.from_dense(vecs[:, 0], [2] * n_g)
print(mps, mps.bond_dims)

wrapped = MPSGSOperator.from_mps(mps, sys_g, permutation=list(block), hamiltonian=H_g)
print(wrapped.expect(H_g), vals[0])

## 6. Fermions

The fermionic models of QALMA whose sites are products of two-level modes (spinless fermions, Hubbard, Kondo) are mapped to qubits with the Jordan-Wigner transformation, so their operators are Pauli strings and `MPSGSOperator` works with them as with any spin system. Hamiltonian terms and observables that are products of local operators are never written as dense matrices: the MPO is built from their local factors, and the expectation values are contracted with the MPS, so strings over many sites are cheap.

For free fermions the exact solution is known: fill the single-particle levels with negative energy. We compare 32 spinless fermions on an open chain, a size far beyond exact diagonalization.

In [ ]:
from qalma.operators.pauli import fermion_annihilation, fermion_creation

n_f = 32
fsys = build_system("open chain lattice", "spinless fermions", L=n_f, t=1, V=0, mu=0.3)
H_f = fsys.global_operator("Hamiltonian")

start = time.time()
rho_f = MPSGSOperator(H_f, permutation=[f"1[{i}]" for i in range(n_f)], chi_max=8)
print("DMRG energy:", rho_f.energy, " (%.1f s)" % (time.time() - start))

# Exact solution of free fermions
position = {site: k for k, site in enumerate(fsys.sites)}
h_single = -0.3 * np.eye(n_f)
for edges in fsys.spec["graph"].edges.values():
    for src, dst in edges:
        h_single[position[src], position[dst]] -= 1
        h_single[position[dst], position[src]] -= 1
levels, modes = np.linalg.eigh(h_single)
occupied = modes[:, levels < 0]
correlations = occupied @ occupied.T
print("exact energy:", levels[levels < 0].sum())

In [ ]:
# <c^+_i c_j>: a Pauli string with a string of sigma^z between the sites
for i, j in ((0, 1), (2, 27), (0, 31)):
    c_dag = fermion_creation(fsys, f"1[{i}]")
    c_ann = fermion_annihilation(fsys, f"1[{j}]")
    value = rho_f.expect(c_dag * c_ann)
    exact = correlations[position[f"1[{i}]"], position[f"1[{j}]"]]
    print(f"<c+_{i} c_{j}> = {np.real(value): .8f}   exact = {exact: .8f}")

A Hubbard chain: every lattice site becomes two qubits, `1[i]:Nup` and `1[i]:Ndown`. With 4 sites (8 qubits) we can compare with exact diagonalization. The order of the qubits along the MPS changes the cost, not the result.

In [ ]:
hub = build_system("open chain lattice", "fermion Hubbard", L=4, t=1, U=4, mu=2)
H_h = hub.global_operator("Hamiltonian")
e_hub = np.linalg.eigvalsh(H_h.to_qutip(tuple(sorted(hub.sites))).full())[0]
print("exact energy:", e_hub)

for label, perm in (("lexicographic", None), ("reversed", list(range(7, -1, -1)))):
    state = MPSGSOperator(H_h, permutation=perm, chi_max=8)
    print(f"{label:14s} E0 = {state.energy:.10f}   order starts with {state.order[:3]}")

## 7. Saving and loading

The state is stored in HDF5 files with the tensors of the MPS, the order of the sites and the Hamiltonian. No dense matrix is written and nothing is stored as a pickle.

In [ ]:
import os
import tempfile

from qalma.io import load_operators, save_operators

filename = os.path.join(tempfile.mkdtemp(), "ground_state.h5")
save_operators(filename, {"rho": gs, "H": H_g})
print(os.path.getsize(filename), "bytes")

loaded = load_operators(filename)
rho_loaded = loaded["rho"]
print(rho_loaded, "- solved:", rho_loaded.solved)

loaded_system = rho_loaded.system
obs_loaded = loaded_system.site_operator("Sz", "1[3]") * loaded_system.site_operator("Sz", "1[4]")
print(np.real(rho_loaded.partial_trace(sites).expect(obs_loaded)))